# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## 1. Setup & Imports

In [1]:
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 2. Hyperparameters

In [8]:
BATCH_SIZE = 64
NUM_EPOCHS = 10

## 3. Data Loading

In [9]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

classes: list[str] = train_set.classes
print("Classes:", classes)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## 4. Model Definition

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [10]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(self, num_classes: int = 10, activation: nn.Module = nn.LeakyReLU()) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.act  = activation
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

## 5. Training & Evaluation Helpers

`train_one_epoch` and `validate` each handle a single pass through their respective loader.  
`fit` orchestrates the loop, tracks history, and restores the best checkpoint.

In [11]:
criterion = nn.CrossEntropyLoss()


def train(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> tuple[float, float]:
    """Run one full pass over `loader` in training mode.

    Returns
    -------
    train_loss : float  – mean cross-entropy loss over all samples
    train_acc  : float  – accuracy in percent
    """
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct      += (outputs.argmax(dim=1) == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def validate(
    model: nn.Module,
    loader: DataLoader,
) -> tuple[float, float]:
    """Run one full pass over `loader` in evaluation mode.

    Returns
    -------
    val_loss : float  – mean cross-entropy loss over all samples
    val_acc  : float  – accuracy in percent
    """
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            correct      += (outputs.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def fit(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = NUM_EPOCHS,
) -> dict[str, list[float]]:
    """Train `model` for `num_epochs`, validating after every epoch.

    Saves the weights that achieved the best validation accuracy and
    restores them at the end.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """
    best_val_acc = 0.0
    best_state   = None
    history: dict[str, list[float]] = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
    }

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train(model, train_loader, optimizer)
        val_loss, val_acc = validate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:>{len(str(num_epochs))}}/{num_epochs} | "
            f"train loss {train_loss:.4f}, train acc {train_acc:.2f}% | "
            f"val loss {val_loss:.4f}, val acc {val_acc:.2f}%"
        )

    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best weights (val acc {best_val_acc:.2f}%)")

    return history

## 6. Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [13]:
model_a    = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=1e-2)

print("=== Experiment A: SGD + LeakyReLU ===")
history_a = fit(model_a, optimizer_a, train_loader, test_loader)

acc_sgd_leaky = history_a["val_acc"][-1]
print(f"\nSGD + LeakyReLU final test accuracy: {acc_sgd_leaky:.2f}%")

=== Experiment A: SGD + LeakyReLU ===
Epoch  1/10 | train loss 2.2566, train acc 16.85% | val loss 2.0868, val acc 26.06%
Epoch  2/10 | train loss 1.9197, train acc 31.41% | val loss 1.8574, val acc 31.58%
Epoch  3/10 | train loss 1.6787, train acc 39.50% | val loss 1.6710, val acc 39.23%
Epoch  4/10 | train loss 1.5282, train acc 44.34% | val loss 1.4710, val acc 46.52%
Epoch  5/10 | train loss 1.4209, train acc 48.38% | val loss 1.5476, val acc 44.09%
Epoch  6/10 | train loss 1.3448, train acc 51.46% | val loss 1.3455, val acc 50.46%
Epoch  7/10 | train loss 1.2797, train acc 54.17% | val loss 1.3710, val acc 50.61%
Epoch  8/10 | train loss 1.2204, train acc 56.64% | val loss 1.1996, val acc 56.69%
Epoch  9/10 | train loss 1.1635, train acc 58.73% | val loss 1.2150, val acc 56.85%
Epoch 10/10 | train loss 1.1119, train acc 60.83% | val loss 1.1390, val acc 59.21%

Restored best weights (val acc 59.21%)

SGD + LeakyReLU final test accuracy: 59.21%


## 7. Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [ ]:
model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=1e-3)

print("=== Experiment B: Adam + LeakyReLU ===")
history_b = fit(model_b, optimizer_b, train_loader, test_loader)

acc_adam_leaky = history_b["val_acc"][-1]
print(f"\nAdam + LeakyReLU final test accuracy: {acc_adam_leaky:.2f}%")

=== Experiment B: Adam + LeakyReLU ===


## 8. Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [ ]:
model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=LR_ADAM)

print("=== Experiment C: Adam + Tanh ===")
history_c = fit(model_c, optimizer_c, train_loader, test_loader)

acc_adam_tanh = history_c["val_acc"][-1]
print(f"\nAdam + Tanh final test accuracy: {acc_adam_tanh:.2f}%")

## 9. Results Summary

In [ ]:
print(f"{'Experiment':<30} {'Test Accuracy':>15}")
print("─" * 47)
print(f"{'SGD  + LeakyReLU':<30} {acc_sgd_leaky:>14.2f}%")
print(f"{'Adam + LeakyReLU':<30} {acc_adam_leaky:>14.2f}%")
print(f"{'Adam + Tanh':<30} {acc_adam_tanh:>14.2f}%")